# Afrobarometer Analysis: Urban/Rural Influences on Democratic Attitudes
## Notebook 1: Data Cleaning
This notebook loads the raw Afrobarometer Round 9 dataset, inspects 
the data, handles missing values, and prepares a clean analytical 
dataset for exploratory analysis.

### Section 1: Import Libraries

In [4]:
# Import libraries
import pandas as pd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
print("Libraries loaded successfully")

Libraries loaded successfully


### Section 2: Load and Check Data

In [33]:
# Load raw Afrobarometer dataset (.sav format)
df, meta = pyreadstat.read_sav('data/raw/AB_r9.sav')

# Check basic shape
print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns")
df.head()

# Check column names
print(df.columns.tolist())

print(df.describe())

# Check data types
df.dtypes

Dataset contains 53444 rows and 421 columns
['RESPNO', 'COUNTRY', 'COUNTRY.old.spelling', 'COUNTRY_R7List', 'COUNTRY_R6List', 'COUNTRY_R5List', 'COUNTRY.BY.REGION', 'URBRUR', 'URBRUR_COND', 'REGION', 'LOCATION.LEVEL.1', 'EA_SVC_A', 'EA_SVC_B', 'EA_SVC_C', 'EA_SVC_D', 'EA_SVC_E', 'EA_FAC_A', 'EA_FAC_B', 'EA_FAC_C', 'EA_FAC_D', 'EA_FAC_E', 'EA_FAC_F', 'EA_FAC_F2', 'EA_FAC_G', 'EA_SEC_A', 'EA_SEC_B', 'EA_SEC_C', 'EA_SEC_D', 'EA_SEC_E', 'EA_ROAD_A', 'EA_ROAD_B', 'EA_ROAD_C', 'NOCALL_1', 'NOCALL_1OTHER', 'NOCALL_2', 'NOCALL_2OTHER', 'NOCALL_3', 'NOCALL_3OTHER', 'NOCALL_4', 'NOCALL_4OTHER', 'NOCALL_5', 'NOCALL_5OTHER', 'NOCALL_6', 'NOCALL_6OTHER', 'NOCALL_7', 'NOCALL_7OTHER', 'NOCALL_8', 'NOCALL_8OTHER', 'NOCALL_9', 'NOCALL_9OTHER', 'NOCALL_10', 'NOCALL_10OTHER', 'NOCALL_11', 'NOCALL_11OTHER', 'NOCALL_12', 'NOCALL_12OTHER', 'NOCALL_13', 'NOCALL_13OTHER', 'NOCALL_14', 'NOCALL_15', 'NOCALL_16', 'NOCALL_17', 'NOCALL_18', 'NOCALL_19', 'NOCALL_20', 'NOCALL_21', 'DATEINTR', 'STRTIME', 'Q1', 'Q2', 

RESPNO                   object
COUNTRY                 float64
COUNTRY.old.spelling    float64
COUNTRY_R7List          float64
COUNTRY_R6List          float64
                         ...   
R4.only                 float64
R5.only                 float64
R6.only                 float64
R7.only                 float64
R8.only                 float64
Length: 421, dtype: object

### Section 3: Selecting Key Variables

In [43]:
# Relevant variables
## IV
### URBRUR = urban/peri-urban/rural classification

## DV
### Q31 = Satisfaction with democracy
### Q23 = Preference for democracy over other systems
### Q30 = Extent of democracy

## Controls
### Q1 = Age
### Q100 = Gender
### Q94 = Education level
### Q93A = Employment status
### Q8 = Discuss politics
### Wealth index
#### Q90 = Which of these things do you personally own?
##### Q90A = Radio
##### Q90B = Television
##### Q90C = Motor vehicle
##### Q90D = Own computer
##### Q90E = Mobile phone

key_vars = ['COUNTRY', 'URBRUR', 'Q31', 'Q23', 'Q30', 'Q1', 'Q100', 'Q94', 'Q93A', 'Q8', 'Q90A', 'Q90B', 'Q90C', 'Q90D', 'Q90E']

df_clean = df[key_vars].copy()

### Section 4: Recode and clean

In [47]:
df_clean.rename(columns={
    'URBRUR': 'urban_rural',
    'Q31': 'satis_demo',
    'Q23': 'prefer_demo',
    'Q30' : 'extent_demo',
    'COUNTRY': 'country',
    'Q1': 'age',
    'Q100' : 'gender',
    'Q94': 'education',
    'Q93A': 'employment',
    'Q90A' : 'radio',
    'Q90B' : 'tv',
    'Q90C' : 'vehicle',
    'Q90D' : 'pc',
    'Q90E' : 'cell',
}, inplace=True)

cols_to_clean = ['satis_demo', 'prefer_demo', 'extent_demo', 'country', 'age', 'gender', 
                 'education', 'employment', 'radio', 'tv', 'vehicle', 'pc', 'cell']

In [45]:
# Cleaning IV: Rural/Urban
## Recode urban_rural into dummy:
### 1 = Urban (urban + peri-urban merged)
### 0 = Rural
df_clean['urban_rural'] = df_clean['urban_rural'].map({
    1: 1,
    2: 0, 
    3: 1
})
print(df_clean['urban_rural'].value_counts())

urban_rural
0    27984
1    25460
Name: count, dtype: int64


In [65]:
# Cleaning DV: Satisfaction with democracy
## Check variable:
### 0=[country] is not a democracy, 
### 1=Not at all satisfied, 
### 2=Not very satisfied, 
### 3=Fairly satisfied, 
### 4=Very satisfied
print(df_clean['satis_demo'].value_counts().sort_index())

## Handle NA's:
### 8 = Refused, 
### 9 = Don’t know, 
### 0 = dont believe country is democracy
df_clean['satis_demo'] = df_clean['satis_demo'].replace([0, 8, 9], np.nan)

## Create dummy:
### 1 = satisfied democracy, 0 = not satisfied democracy
df_clean['satis_dem_dummy'] = np.where(
    df_clean['satis_demo'].isna(), np.nan, 
    (df_clean['satis_demo'].isin([3, 4])).astype(int))

print(df_clean['satis_dem_dummy'].value_counts().sort_index())

satis_demo
1.0    13907
2.0    16579
3.0    14764
4.0     6011
Name: count, dtype: int64
satis_dem_dummy
0.0    30486
1.0    20775
Name: count, dtype: int64


In [70]:
# Cleaning DV: Prefer democracy over any other system
## Check variable:
### 1 = Doesn’t matter, 
### 2 = Sometimes non-democratic preferable, 
### 3 = Democracy preferable
print(df_clean['prefer_demo'].value_counts().sort_index())

## Handle NA's:
### 8 = Refused, 
### 9 = Don’t know
df_clean['prefer_demo'] = df_clean['prefer_demo'].replace([8, 9], np.nan)

## Create dummy:
### 1 = prefer democracy, 
### 0 = non-democratic preferable/doesn't matter
df_clean['prefer_dem_dummy'] = np.where(
    df_clean['prefer_demo'].isna(), np.nan, 
    (df_clean['prefer_demo'] == 3).astype(int))

print(df_clean['prefer_dem_dummy'].value_counts().sort_index())

prefer_demo
1.0     8103
2.0     8043
3.0    35896
Name: count, dtype: int64
prefer_dem_dummy
0.0    16146
1.0    35896
Name: count, dtype: int64


In [80]:
# Cleaning DV: Extent of democracy
## Check variable:
### 1 = Not a democracy, 
### 2 = A democracy, with major problems, 
### 3 = A democracy, but with minorproblems, 
### 4 = A full democracy, 
print(df_clean['extent_demo'].value_counts().sort_index())

## Handle NA's:
### 8 = Do not understand question/do not understand what ‘democracy’ is,
### 9 = Don’t know, 
### 98 = Refused
df_clean['extent_demo'] = df_clean['extent_demo'].replace([8, 9, 98], np.nan)

## Create dummy:
df_clean['extent_dem_dummy'] = np.where(
    df_clean['extent_demo'].isna(), np.nan, 
    (df_clean['extent_demo'] == 4).astype(int))

print(df_clean['extent_dem_dummy'].value_counts().sort_index())

extent_demo
1.0     8704
2.0    18046
3.0    16675
4.0     8285
Name: count, dtype: int64
extent_dem_dummy
0.0    43425
1.0     8285
Name: count, dtype: int64


In [89]:
# Cleaning controls: Age
## Handle NA's:
### 998 = Refused
### 999 = Don’t know
df_clean['age'] = df_clean['age'].replace([998, 999], np.nan)

## Create youth dummy:
### African Union consider anyone between 15 and 35 as youth
### 1 = youth (18-34)
### 0 = not youth (35 upwards
df_clean['youth'] = np.where(
    df_clean['age'].isna(), np.nan,
    (df_clean['age'] < 35).astype(int))

print(df_clean['youth'].value_counts(dropna=False))
print(df_clean.groupby('youth')['age'].describe())

youth
0.0    27660
1.0    25724
NaN       60
Name: count, dtype: int64
         count       mean        std   min   25%   50%   75%    max
youth                                                              
0.0    27660.0  49.515221  11.873474  35.0  40.0  47.0  57.0  112.0
1.0    25724.0  25.690833   4.731710  18.0  22.0  25.0  30.0   34.0


In [89]:
# Cleaning controls: Age
## Handle NA's:
### 998 = Refused
### 999 = Don’t know
df_clean['age'] = df_clean['age'].replace([998, 999], np.nan)

## Create youth dummy:
### African Union consider anyone between 15 and 35 as youth
### 1 = youth (18-34)
### 0 = not youth (35 upwards
df_clean['youth'] = np.where(
    df_clean['age'].isna(), np.nan,
    (df_clean['age'] < 35).astype(int))

print(df_clean['youth'].value_counts(dropna=False))
print(df_clean.groupby('youth')['age'].describe())

youth
0.0    27660
1.0    25724
NaN       60
Name: count, dtype: int64
         count       mean        std   min   25%   50%   75%    max
youth                                                              
0.0    27660.0  49.515221  11.873474  35.0  40.0  47.0  57.0  112.0
1.0    25724.0  25.690833   4.731710  18.0  22.0  25.0  30.0   34.0


In [92]:
# Cleaning controls: Education level
### 0-9 from 0 being no formal schooling to 9 being post-graduate
print(df_clean['education'].value_counts().sort_index())

## Handle NA's:
### 98 = Refused
### 99 = Don’t know
df_clean['education'] = df_clean['education'].replace([98, 99], np.nan)

print(df_clean['education'].describe())

education
0.0     7560
1.0     2038
2.0     7801
3.0     7341
4.0    10612
5.0     8764
6.0     3131
7.0     2400
8.0     3123
9.0      533
Name: count, dtype: int64
count    53303.000000
mean         3.588879
std          2.271482
min          0.000000
25%          2.000000
50%          4.000000
75%          5.000000
max          9.000000
Name: education, dtype: float64


In [102]:
# Cleaning controls: Employment
## Check variable:
### 0 = No (not looking), 
### 1 = No (looking), 
### 2 = Yes (part time), 
### 3 = Yes (full time)
print(df_clean['employment'].value_counts().sort_index())

## Handle NA's:
### 8 = Refused
### 9 = Don’t know
df_clean['employment'] = df_clean['employment'].replace([8, 9, 9994], np.nan)

## Employment dummy:
df_clean['employment_dummy'] = np.where(
    df_clean['employment'].isna(), np.nan,
    (df_clean['employment'].isin([2, 3])
).astype(int))

print(df_clean['employment_dummy'].value_counts().sort_index())

employment
0.0    18663
1.0    16206
2.0     5623
3.0    11698
Name: count, dtype: int64
employment_dummy
0.0    34869
1.0    17321
Name: count, dtype: int64


In [109]:
# Cleaning controls: Wealth Index
## Check variables: each item is coded from 0-2 as follows:
### 0 = No one in household owns, 
### 1 = Yes someone else in household owns, 
### 2 = Yes personally owns
print(df_clean['radio'].value_counts().sort_index())
print(df_clean['tv'].value_counts().sort_index())
print(df_clean['vehicle'].value_counts().sort_index())
print(df_clean['pc'].value_counts().sort_index())
print(df_clean['cell'].value_counts().sort_index())

## Handle NA's:
### 8 = Refused
### 9 = Don’t know
asset_vars = ['radio', 'tv', 'vehicle', 'pc', 'cell']
df_clean[asset_vars] = df_clean[asset_vars].replace([8, 9], np.nan)

## Creating wealth index:
### Index goes from 0-10 with 0 being low wealth and 10 being high wealth
df_clean['wealth_index'] = df_clean[['radio', 'tv', 
                                      'vehicle', 'pc', 
                                      'cell']].sum(axis=1)

print(df_clean['wealth_index'].value_counts().sort_index())

radio
0.0    13193
1.0     9096
2.0    31048
Name: count, dtype: int64
tv
0.0    18934
1.0     7649
2.0    26740
Name: count, dtype: int64
vehicle
0.0    29877
1.0    11656
2.0    11720
Name: count, dtype: int64
pc
0.0    36605
1.0     7693
2.0     8836
Name: count, dtype: int64
cell
0.0    24108
1.0     8015
2.0    20674
Name: count, dtype: int64
wealth_index
0.0     5781
1.0     2058
2.0     8883
3.0     3098
4.0     8254
5.0     3830
6.0     7956
7.0     3388
8.0     5361
9.0     1829
10.0    3006
Name: count, dtype: int64


In [110]:
print(df_clean.groupby('urban_rural')['wealth_index'].mean())

urban_rural
0    3.556103
1    5.602160
Name: wealth_index, dtype: float64


In [115]:
# Drop NA's:
df_clean = df_clean.dropna(subset = ['satis_demo', 
                                     'prefer_demo', 
                                     'extent_demo', 
                                     'country', 
                                     'age', 
                                     'gender', 
                                     'education', 
                                     'employment', 
                                     'radio', 
                                     'tv', 
                                     'vehicle', 
                                     'pc', 
                                     'cell'])

print(f"Clean dataset contains {len(df_clean)} rows and {df_clean.shape[1]} columns")
print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns")

Clean dataset contains 47872 rows and 21 columns
Dataset contains 53444 rows and 421 columns


In [119]:
print(f"Original dataset: {len(df)} rows")
print(f"Clean dataset: {len(df_clean)} rows")
print(f"Rows dropped: {len(df) - len(df_clean)}")
print(f"Percentage kept: {round(len(df_clean)/len(df)*100, 2)}%")

Original dataset: 53444 rows
Clean dataset: 47872 rows
Rows dropped: 5572
Percentage kept: 89.57%


### Section 5: Save clean dataset:

In [120]:
df_clean.to_csv('data/clean_ab.csv', index=False)
print("Clean dataset saved successfully")

Clean dataset saved successfully


### Section 6: Summary:

- Original dataset: 421 columns and 53444 rows
- Key variables selected for analysis:
  Independent variable:
  URBRUR = urban/peri-urban/rural classification

  Dependent variable:
  Q31 = Satisfaction with democracy
  Q23 = Preference for democracy over other systems
  Q30 = Extent of democracy

  Control variables:
  Q1 = Age
  Q100 = Gender
  Q94 = Education level
  Q93A = Employment status
  Wealth index
  Q90 = Which of these things do you personally own?
  Q90A = Radio
  Q90B = Television
  Q90C = Motor vehicle
  Q90D = Own computer
  Q90E = Mobile phone

- Recoded missing value codes to NaN for each variable
- Created dummy variables for: satis_demo, prefer_demo, youth, urban_rural
- Created wealth index from 5 asset variables (range 0-10)
- Dropped rows with missing values in key variables
- Final clean dataset: 21 columns and 47872 rows
- Saved to data/clean_ab.csv